Importing Dependencies

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, ServiceContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import wikipediaapi
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from llama_index.core.node_parser import SentenceSplitter

Set up Wikipedia API and fetch articles

In [ ]:
wiki = wikipediaapi.Wikipedia("en")

animals = ["Elephant", "Cheetah", "Giraffe", "Dolphin", "Penguin"]

data = []
for animal in animals:
    page = wiki.page(animal)
    if page.exists():
        text = page.text
        data.append({"animal": animal, "text": text})
        with open(f"docs/{animal}.txt", "w", encoding="utf-8") as f:
            f.write(text)

Simple test with model

In [ ]:
import ollama

response = ollama.chat(model='llama2', messages=[
  {'role': 'user', 'content': 'Explain retrieval augmented generation in one sentence.'}
])

print(response['message']['content'])

In [ ]:
model_id = "meta-llama/Llama-2-7b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    load_in_4bit=True
)

prompt = "Explain retrieval augmented generation in one sentence."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Load the model in with its tokenizer

In [ ]:
model_id = "meta-llama/Llama-2-7b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True
)

# Wrap it for llamaindex
llm = HuggingFaceLLM(
    model=model,
    tokenizer=tokenizer,
    generate_kwargs={"max_new_tokens": 150},
)

Helper function to generate responses for different chunk sizes

In [ ]:
def generate_responses(query):
    documents = SimpleDirectoryReader("docs").load_data()

    chunk_sizes = [128, 256, 512, 1024]

    embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

    for chunk_size in chunk_sizes:
        print(f"\n=== Chunk size: {chunk_size} ===")

        # Create a sentence-based text splitter
        splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=50) 

        service_context = ServiceContext.from_defaults(
            llm=llm,
            embed_model=embed_model,
            text_splitter=splitter
        )

        index = VectorStoreIndex.from_documents(documents, service_context=service_context)
        query_engine = index.as_query_engine()

        response = query_engine.query(query)
        print("Response:", response.response)


Testing RAG retrieval for different queries

In [ ]:
generate_responses("Explain how lions hunt together.")
generate_responses("What do elephants eat?")
generate_responses("Describe the habitat of penguins.")
generate_responses("How fast can a cheetah run?")